# 信贷违约预测：完整项目流程与最终报告

按照“测试评估与泛化—业务解释与阈值—局限与改进”组织最终交付。
数据源、特征规则和最终模型来自本项目，不引入外部数据。

**仓库运行边界**：当前可阅读代码与报告；GitHub 版本已清理执行输出，且未提供原始 CSV、清洗快照和最终模型等完整重放输入。下述 results 路径为原流程的本地输入或生成目录，不表示仓库已包含相应文件；缺少输入时请勿直接从头执行。
**最终结果口径**：90 特征、按完整放款月份划分、测试集 79,898 条；基线 AUC 0.7220，调优模型 AUC 0.7198。早期随机划分、335 特征的 AUC 0.7330 不是本流程的最终成绩；本次文档整理未重训或重算。

**两种运行模式**：默认 `RUN_TRAINING=False` 完整重放原始清洗、核对数据、加载最终模型、重算评估与生成报告；
改为 `True` 则额外重跑 5 折时间交叉验证、20 次 Optuna 搜索、验证期模型选择及合并再训练，可能耗时较长。
原 `results/xgboost_optimization` 不会被覆盖，重训输出进入时间戳目录。

运行前选择项目 `venv/Scripts/python.exe` 对应内核，并保留 `code/model` 中的自定义预处理和报告模块。
本 notebook 从项目根目录、code 目录或其子目录启动均可；更换位置时应显式设置 PROJECT_DIR。
**报告不表示模型已上线**：新阈值是回顾性候选，没有新增时间外样本的正式经营验收。

## 0. 环境与运行配置

随机种子 42；时间切点为 2016-03-31、2017-06-30。只加载可信本地 joblib。
运行依赖与文件摘要会保存至 `run_manifest.json`；依赖版本以现有项目环境为准，不在 notebook 中自动安装或升级。

In [ ]:
from pathlib import Path
from functools import partial
import gc
import json
import sys
import joblib
import numpy as np
import pandas as pd
import optuna
from IPython.display import display, Image

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next((path for path in candidates if (path / "data" / "credit_data(1).csv").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("请从 D:/credit 或其子目录启动，或显式设置 PROJECT_DIR。")
sys.path.insert(0, str(PROJECT_DIR / "code" / "model"))
import xgboost_optimization as optimization
import final_project as final

RUN_TRAINING = False
N_TRIALS = 20
BOOTSTRAP_REPETITIONS = 300
PREDICTION_BATCH_SIZE = 4000
SEED = 42
TARGET = "isDefault"
RUN_ID = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
SOURCE_DIR = PROJECT_DIR / "results" / "xgboost_optimization"
OUTPUT_DIR = PROJECT_DIR / "results" / "final_project"
if RUN_TRAINING:
    SOURCE_DIR = PROJECT_DIR / "results" / "final_pipeline_runs" / RUN_ID / "training"
    OUTPUT_DIR = SOURCE_DIR.parent / "analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)
print("项目：", PROJECT_DIR)
print("运行模式：", "完整重训" if RUN_TRAINING else "既有模型完整重放")
print("结果目录：", OUTPUT_DIR)

## 1. 原始数据检查与确定性清洗

保留原始缺失；工龄 `10+ years`→10、`< 1 year`→0；按明确格式解析日期；负 DTI 或 999 设缺失。
不在切分前使用总体众数、标签或分组统计。只用数值/日期的确定性转换，不把未知日期或工龄静默转为缺失。
不以 id、放款日期直接作模型预测特征，不盲目删除高收入或大额贷款。

In [ ]:
raw = pd.read_csv(PROJECT_DIR / "data" / "credit_data(1).csv")
raw_shape = raw.shape
cleaned = final.clean_raw(raw)
data, cleaning_audit = final.verify_cleaned(cleaned, PROJECT_DIR / "data" / "data_processing.csv", OUTPUT_DIR)
data = data.sort_values(["issueDate", "id"]).reset_index(drop=True)
print(f"原始形状：{raw_shape}；清洗后形状：{data.shape}")
print("清洗重放逐字段不一致总数：", int(cleaning_audit["不一致行数"].sum()))
print("违约标签：", data[TARGET].value_counts().to_dict())
del raw, cleaned
gc.collect()

## 2. 时间切分与仅开发期探索

相同放款月份不跨集合；训练期与验证期是开发集，测试期只作最终评价。
下方探索统计只使用开发期数据，避免依据测试分布反复设计特征。

In [ ]:
train_data = data.loc[data["issueDate"] <= optimization.TRAIN_END]
validation_data = data.loc[(data["issueDate"] > optimization.TRAIN_END) & (data["issueDate"] <= optimization.VALIDATION_END)]
test_data = data.loc[data["issueDate"] > optimization.VALIDATION_END]
development = data.loc[data["issueDate"] <= optimization.VALIDATION_END]
assert train_data["issueDate"].max() < validation_data["issueDate"].min()
assert validation_data["issueDate"].max() < test_data["issueDate"].min()
assert len(train_data) + len(validation_data) + len(test_data) == len(data)
sample_summary = pd.DataFrame([
    {"样本": name, "样本数": len(frame), "违约率": frame[TARGET].mean(),
     "开始日期": str(frame["issueDate"].min().date()), "结束日期": str(frame["issueDate"].max().date())}
    for name, frame in [("训练集", train_data), ("验证集", validation_data), ("测试集", test_data)]
])
display(sample_summary)
missingness = development.isna().mean().sort_values(ascending=False).rename_axis("字段").reset_index(name="开发集缺失率")
display(missingness.head(12))
display(development.groupby("grade", observed=True)[TARGET].agg(["size", "mean"]))
final.save_table(missingness, OUTPUT_DIR, "development_missingness.csv")
final.save_table(sample_summary, OUTPUT_DIR, "sample_summary_replay.csv")
if not RUN_TRAINING:
    historical = pd.read_csv(SOURCE_DIR / "sample_summary.csv")
    pd.testing.assert_frame_equal(sample_summary, historical, check_dtype=False, rtol=1e-12)

## 3. 业务特征、匿名聚合及训练窗口内预处理

流程复用并锁定 `model_optimization_features.py` 与 `xgboost_optimization.py`：

- 单行特征：年度贷款收入比、循环余额收入比、开放账户占比、关闭账户数、信用历史年限、FICO 均值。
- n0–n14 聚合：和、均值、中位数、最小/最大值、总体标准差、非零数、缺失数，在填补前逐行计算。
- 每一训练窗口内：众数填补、缺失率超过 5% 标记、单/双类别交叉统计及标签熵、有序/OneHot/目标编码。
- IV≥0.02，Spearman 相关阈值 0.8，VIF 阈值 10，筛选抽样 20,000；不使用外层验证或测试标签拟合。

注意：目标编码内部是随机交叉拟合，不是严格逐时间历史编码；标签熵的训练记录参与自身分组统计，仍需未来改进。

In [ ]:
example = optimization.build_features(train_data.head(3))
aggregated = optimization.AnonymousFeatureAggregator(optimization.ANONYMOUS_VARIABLES).transform(example)
display(aggregated[["annual_loan_income_ratio", "openAcc_ratio", "ficoRange_mean", "anonymous_nonzero_count", "anonymous_missing_count"]])
assert "isDefault" not in example and "id" not in example and "issueDate" not in example
if RUN_TRAINING:
    optimization.OUTPUT_DIR = SOURCE_DIR
    train_features = optimization.build_features(train_data)
    validation_features = optimization.build_features(validation_data)
    train_target = train_data[TARGET]
    folds, fold_summary = optimization.prepare_folds(train_features, train_target, train_data["issueDate"])
    final.save_table(sample_summary, SOURCE_DIR, "sample_summary.csv")
    final.save_table(fold_summary, SOURCE_DIR, "fold_summary.csv")
else:
    fold_summary = pd.read_csv(SOURCE_DIR / "fold_summary.csv")
display(fold_summary)

## 4. 早期三模型历史与最终时间验证口径

逻辑回归和随机森林是前期选型基线；其旧版随机切分与特征方案不同，仅回顾历史，不据此证明最终模型超过三种算法。
最终严格同口径比较的是基线 XGBoost 与调优 XGBoost，使用相同时间切分和特征处理。

In [ ]:
historical_metrics_path = PROJECT_DIR / "docs" / "model" / "三模型测试集指标.csv"
if historical_metrics_path.exists():
    print("仅为项目历史，不与最终时间外指标混比：")
    display(pd.read_csv(historical_metrics_path))
print("XGBoost 基线参数：", optimization.BASELINE_PARAMETERS)
print("搜索空间：", optimization.SEARCH_SPACE)

## 5. 训练期 5 折验证与 20 次 Optuna 搜索（可选重训）

目标为训练期扩展窗口平均 ROC-AUC 最大化。前 5 次随机启动，此后 TPE 搜索；不借用独立验证或测试标签早停。
默认直接读取已有搜索证据，不能称为本次重新训练。

In [ ]:
if RUN_TRAINING:
    baseline_cv, baseline_oof = optimization.cross_validate("基线_XGBoost", optimization.BASELINE_PARAMETERS, folds)
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=5))
    study.optimize(partial(optimization.objective, folds=folds), n_trials=N_TRIALS)
    best_parameters = study.best_params
    optimized_cv, optimized_oof = optimization.cross_validate("调优_XGBoost", best_parameters, folds)
    cv_metrics = pd.concat([baseline_cv, optimized_cv], ignore_index=True)
    cv_summary = cv_metrics.groupby("模型", sort=False)["AUC"].agg(["mean", "std"]).reset_index()
    cv_summary.columns = ["模型", "CV平均AUC", "CV标准差"]
    best_table = pd.DataFrame([{"模型": "XGBoost", "最佳参数": json.dumps(best_parameters, ensure_ascii=False),
                                "CV平均AUC": study.best_value, "CV标准差": study.best_trial.user_attrs["auc_std"]}])
    trials = pd.DataFrame([{"试验": trial.number, **trial.params, "平均AUC": trial.value,
                            "AUC标准差": trial.user_attrs["auc_std"], "耗时秒": trial.duration.total_seconds()}
                           for trial in study.trials])
    for table, name in [(cv_metrics, "cv_fold_metrics.csv"), (cv_summary, "cv_summary.csv"),
                        (best_table, "best_parameters.csv"), (trials, "trials_XGBoost.csv")]:
        final.save_table(table, SOURCE_DIR, name)
    oof = train_data[["id", "issueDate", TARGET]].join(pd.DataFrame({"基线_XGBoost": baseline_oof, "调优_XGBoost": optimized_oof}))
    final.save_table(oof, SOURCE_DIR, "train_oof_predictions.csv")
    del folds
    gc.collect()
else:
    best_table = pd.read_csv(SOURCE_DIR / "best_parameters.csv")
    best_parameters = json.loads(best_table.iloc[0]["最佳参数"])
    cv_summary = pd.read_csv(SOURCE_DIR / "cv_summary.csv")
display(best_table)
display(cv_summary)

## 6. 独立验证期选型与合并开发集再训练（可选重训）

先根据验证期 AUC、KS 固定模型名称；再合并训练和验证样本，重新拟合全部处理器及两套模型。
测试仅在固定选型后评估；两套测试指标用于事后比较，不反过来选赢家。

In [ ]:
if RUN_TRAINING:
    _, _, train_ready, validation_ready = optimization.fit_features(train_features, train_target, validation_features)
    models = optimization.fit_models(train_ready, train_target, best_parameters)
    validation_probabilities = optimization.predict_models(models, validation_ready)
    validation_metrics = pd.DataFrame([optimization.evaluate(name, validation_data[TARGET], validation_probabilities[name])
                                       for name in validation_probabilities]).sort_values(["AUC", "KS"], ascending=False, kind="stable")
    selected_model = validation_metrics.iloc[0]["模型"]
    final.save_table(validation_metrics, SOURCE_DIR, "validation_metrics.csv")
    final.save_table(validation_data[["id", "issueDate", TARGET]].join(validation_probabilities), SOURCE_DIR, "validation_predictions.csv")
    del train_ready, validation_ready, models
    gc.collect()
    preprocessor, selected_features, development_ready, test_ready = optimization.fit_features(
        optimization.build_features(development), development[TARGET], optimization.build_features(test_data))
    models = optimization.fit_models(development_ready, development[TARGET], best_parameters)
    test_probabilities = optimization.predict_models(models, test_ready)
    test_metrics = pd.DataFrame([optimization.evaluate(name, test_data[TARGET], test_probabilities[name]) for name in test_probabilities])
    test_metrics["相对基线AUC差"] = test_metrics["AUC"] - test_metrics.set_index("模型").loc["基线_XGBoost", "AUC"]
    final.save_table(test_metrics, SOURCE_DIR, "test_metrics.csv")
    final.save_table(test_data[["id", "issueDate", TARGET]].join(test_probabilities), SOURCE_DIR, "test_predictions.csv")
    final.save_table(pd.DataFrame({"特征": selected_features}), SOURCE_DIR, "final_features.csv")
    bundle = {"preprocessor": preprocessor, "selected_features": selected_features, "models": models,
              "selected_model": selected_model, "threshold": 0.5, "parameters": best_parameters}
    joblib.dump(bundle, SOURCE_DIR / "final_model.joblib")
    del development_ready, test_ready
    gc.collect()
else:
    bundle = joblib.load(SOURCE_DIR / "final_model.joblib")
    selected_model = bundle["selected_model"]
assert selected_model == pd.read_csv(SOURCE_DIR / "validation_metrics.csv").sort_values(["AUC", "KS"], ascending=False, kind="stable").iloc[0]["模型"]
assert bundle["selected_features"] == pd.read_csv(SOURCE_DIR / "final_features.csv")["特征"].tolist()
print("验证期固定选型：", selected_model, "；最终特征数：", len(bundle["selected_features"]))
print("标签熵保留数：", sum(name.endswith("_y_entropy") for name in bundle["selected_features"]))

## 7. 模型重加载、全测试概率重放与指标重算

仅当重放概率与历史文件一致时继续生成报告，防止模型、特征或样本错配。
ROC-AUC、KS 与阈值无关；Precision、Recall、F1、混淆矩阵随阈值变化；PR-AUC 采用 average precision。

In [ ]:
predictions = {
    "验证集": pd.read_csv(SOURCE_DIR / "validation_predictions.csv", parse_dates=["issueDate"]),
    "测试集": pd.read_csv(SOURCE_DIR / "test_predictions.csv", parse_dates=["issueDate"]),
}
for split_name, frame, saved in [("验证集", validation_data, predictions["验证集"]), ("测试集", test_data, predictions["测试集"])]:
    pd.testing.assert_frame_equal(frame[["id", "issueDate", TARGET]].reset_index(drop=True), saved[["id", "issueDate", TARGET]], check_dtype=False)
replay_audit = final.replay_predictions(bundle, test_data, predictions["测试集"], OUTPUT_DIR, PREDICTION_BATCH_SIZE)
metrics = final.validate_saved_results(predictions, SOURCE_DIR, OUTPUT_DIR)
display(replay_audit)
display(metrics[["集合", "模型", "AUC", "KS", "PR-AUC", "Precision", "Recall", "F1", "TN", "FP", "FN", "TP"]])

## 8. 泛化分析：不确定性、季度稳定性、风险分层、校准与漂移

两种固定模型配对重采样：逐笔与整月分块，各 300 次。区间不包含模型重训及调参的不确定性。
PSI 按开发集原始字段边界计算；风险十分档用于排序评价，不用测试标签调阈值。

In [ ]:
intervals = final.paired_bootstrap(predictions["测试集"], selected_model, OUTPUT_DIR, BOOTSTRAP_REPETITIONS)
quarterly, risk, calibration = final.stability_tables(predictions["测试集"], selected_model, OUTPUT_DIR)
drift = final.drift_table(development, test_data, OUTPUT_DIR)
display(intervals)
display(quarterly[["季度", "样本数", "违约率", "AUC", "KS"]])
display(risk)
display(drift)

## 9. 业务解读：最终模型关键特征

在开发期 2,000 条固定随机样本上计算 XGBoost 原生 TreeSHAP，无需另装 shap 包。
平均绝对 SHAP 代表依赖度，不能说明因果作用或概率变化百分点；分组统计不能解释成个人原始字段的效应。

In [ ]:
importance, groups, directions = final.explain_model(bundle, development, OUTPUT_DIR, sample_size=2000)
display(importance.head(15))
display(groups)
display(directions)

## 10. 风险阈值：只用验证期提出候选

保留原模型 0.5；新增最大 KS、最大 F1 和两个演示业务约束，不修改模型或在测试集找最优线。
**本次为既有测试已产生之后的回顾性分析**。验证与测试模型经过不同训练规模拟合，绝对概率可能漂移，
候选不能直接升级为生产拒贷线；需要新增校准窗口、成本/产能目标以及新的时间外验证。

In [ ]:
policies, threshold_search = final.threshold_candidates(predictions["验证集"], selected_model, OUTPUT_DIR,
                                                       max_rejection_rate=0.30)
policy_table = final.evaluate_policies(predictions, selected_model, policies, OUTPUT_DIR)
display(policy_table[["策略", "集合", "阈值", "Precision", "Recall", "标记比例", "通过组违约率", "F1"]])
bundle = final.apply_business_threshold(bundle, predictions, policies, OUTPUT_DIR)
print("已应用业务约束阈值：", bundle["threshold"], "；模型与分类结果另存于：", OUTPUT_DIR)

## 11. 图表、项目最终报告与完整性台账

报告按照用户指定的三个章节自动填入本次重算数值，生成 Markdown 与可离线浏览的 HTML。
不宣称已完成利润验证、因果识别、独立校准、当前新客群测试或上线验收。

In [ ]:
final.make_figures(predictions["测试集"], selected_model, quarterly, risk, calibration, importance, threshold_search, OUTPUT_DIR)
manifest = final.write_manifest(PROJECT_DIR, SOURCE_DIR, OUTPUT_DIR, bundle, policies, BOOTSTRAP_REPETITIONS)
manifest["training_executed_this_run"] = RUN_TRAINING
(OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
report_path = final.write_report(PROJECT_DIR, SOURCE_DIR, OUTPUT_DIR, bundle, metrics, policy_table,
                                 intervals, quarterly, risk, calibration, importance, groups, directions, drift, manifest)
display(Image(filename=str(OUTPUT_DIR / "test_evaluation.png")))
display(Image(filename=str(OUTPUT_DIR / "business_analysis.png")))
print("报告：", report_path)
print("HTML：", report_path.with_suffix(".html"))
print("证据及图表：", OUTPUT_DIR)

## 12. 已完成范围、局限及下一步

- 默认模式完成数据与预测全量重放，训练分支只有开启 RUN_TRAINING 才会执行；不能将读取已有文件称为重新训练。
- 当前模型定位为离线风险排序/人工复核原型，既有 0.5 阈值可能遗漏大量违约，且调优不保证超过基线。
- 优先核对输入形成时点、既有评级依赖、违约标签成熟度、已放款样本选择及匿名字段含义。
- 业务确认成本与复核容量后，新增校准/阈值窗口，锁定模型和策略，用新时间外样本一次性验收。
- 更详细的数量结论、关键特征解释与 P0/P1/P2 改进任务见自动生成的最终项目报告。